# 03 · Feature engineering

**Goal:** test the engineered feature ideas from the EDA, one block at a time, and keep only those that help. Each block is compared with the 37-feature set from `02_feature_selection` on the fixed folds (5-fold × 3 seeds), fold by fold, for **both** LightGBM and CatBoost.

The feature code lives in `models/features.py`, so later notebooks reuse exactly the same definitions.

| Block | Features | EDA motivation |
|---|---|---|
| **A1 ZIP** | ZIP and ZIP3 | neighborhood effects beyond the Census features |
| **A2 combos** | prior carrier × state, coverage package × CAT zone | carrier effect (7 pp) may vary by state; package and CAT zone stack (16% vs 7%) |
| **A3 make/model** | 1,049 vehicle models, encoded per vehicle → mean/max per policy | make showed only a luxury effect; does the model add anything? |
| **B household** | age 35–50 flag, age spread, # young drivers, drivers per vehicle | driver age acts as a band, not a slope |
| **C price** | quote per driver, per vehicle, per credit point | low-credit customers are the price-sensitive ones |
| **D geography** | ZIP latitude/longitude (Census Gazetteer) | regional patterns beyond state and density |
| **E bundle** | 0 = no discount / 1 = discount only / 2 = home + discount | home policy always comes with a discount |

**How block A is fed to each model:**
- **LightGBM** gets a smoothed **target encoding** (`TE_*`): conversion rate per level, shrunk toward the overall rate with weight 20. It's learned **inside each fold** from the training part only, and training rows are themselves encoded out-of-fold (inner 5-fold), so no row ever sees its own target.
- **CatBoost** gets the raw keys as categoricals, since its built-in ordered target statistics do the same job.
- **Make/model** has several values per policy, so both models get the in-fold encoding, rolled up per policy.

**Decision rule:** keep a block if it improves at least one model by more than 2 standard errors (paired) and doesn't hurt the other by more than 2 standard errors.

In [1]:
import json, sys
import numpy as np, pandas as pd
sys.path.insert(0, '.')
import common as C, features as FE

lab, hold = C.load()
lab = FE.add_static(lab)
KEEP = json.load(open('features_selected.json'))['features']
fmt = {'auc_mean': '{:.4f}', 'auc_se': '{:.4f}', 'Δ auc': '{:+.4f}', 'se Δ': '{:.4f}', 'top_decile_prec': '{:.1%}'}
base = {m: C.cv_score(f'fe_base_{m}', lab, KEEP, model=m, notes='feature engineering: 37 selected features') for m in ('lgb', 'cb')}
print({m: round(r['auc_mean'], 4) for m, r in base.items()})

{'lgb': np.float64(0.6956), 'cb': np.float64(0.7011)}


## 1. One block at a time
For each block and model: the features to add, plus a `fold_fn` when a target encoding is needed.

In [2]:
def spec(block, model):
    """(extra features, fold_fn) for a block on a given model."""
    if block in FE.TE_KEYS:
        keys = FE.TE_KEYS[block]
        return (keys, None) if model == 'cb' else ([f'TE_{k}' for k in keys], FE.te_fold_fn(keys))
    if block == 'A3_make_model':
        return FE.MM_COLS, FE.te_fold_fn(make_model=True)
    return FE.BLOCKS[block], None

ALL = list(FE.TE_KEYS) + ['A3_make_model'] + list(FE.BLOCKS)
runs = {m: {} for m in base}
for b in ALL:
    for m in base:
        extra, fn = spec(b, m)
        runs[m][b] = C.cv_score(f'fe_{b}_{m}', lab, KEEP + extra, model=m, fold_fn=fn, notes=f'feature engineering: + {b}')

COLS = ['auc_mean', 'Δ auc', 'se Δ', 'top_decile_prec']
tbl = pd.concat([C.summary([base[m]] + list(runs[m].values()), baseline=base[m]['name']).iloc[1:][COLS]
                 .rename(index=lambda n: n.removeprefix('fe_').rsplit('_', 1)[0]).add_prefix(f'{m} ') for m in base], axis=1)
z = {m: tbl[f'{m} Δ auc'] / tbl[f'{m} se Δ'] for m in base}  # Δ in standard errors
tbl['keep'] = ((z['lgb'] > 2) | (z['cb'] > 2)) & (z['lgb'] > -2) & (z['cb'] > -2)
tbl.style.format({f'{m} {c}': f for m in base for c, f in fmt.items() if c in COLS})

,lgb auc_mean,lgb Δ auc,lgb se Δ,lgb top_decile_prec,cb auc_mean,cb Δ auc,cb se Δ,cb top_decile_prec,keep
name,,,,,,,,,
A1_zip,0.6939,-0.0017,0.0004,28.1%,0.7010,-0.0001,0.0005,28.2%,False
A2_combos,0.6949,-0.0008,0.0005,27.6%,0.7009,-0.0002,0.0005,28.5%,False
A3_make_model,0.6938,-0.0018,0.0004,27.8%,0.6994,-0.0017,0.0004,27.9%,False
B_household,0.7049,+0.0092,0.0009,28.7%,0.7078,+0.0067,0.0011,29.2%,True
C_price,0.6947,-0.0010,0.0003,27.6%,0.7005,-0.0006,0.0007,28.0%,False
D_geo,0.6944,-0.0012,0.0004,28.1%,0.6997,-0.0014,0.0005,28.2%,False
E_bundle,0.6950,-0.0007,0.0003,27.9%,0.6998,-0.0013,0.0006,28.0%,False


**Findings** (paired Δ vs the 37-feature base; z = Δ ÷ its standard error)

| Block | LightGBM Δ | CatBoost Δ | Verdict |
|---|---|---|---|
| **B household** | **+0.0092** (z 10.6) | **+0.0067** (z 6.0) | **keep** |
| A1 ZIP | −0.0017 (z −4.3) | −0.0001 | drop |
| A2 combos | −0.0008 | −0.0002 | drop |
| A3 make/model | −0.0018 (z −4.7) | −0.0017 (z −4.1) | drop |
| C price | −0.0010 (z −3.0) | −0.0006 | drop |
| D geography | −0.0012 (z −3.0) | −0.0014 (z −2.8) | drop |
| E bundle | −0.0007 (z −2.5) | −0.0013 (z −2.3) | drop |

- **Only the household block helps, and by a lot:** it's the largest single gain since the baseline.
- **The target encodings don't help,** even though package × CAT zone and carrier × state are strong on their own (AUC ≈ 0.58). The trees already learn those interactions from the raw columns, and ZIP / make-model add noise. The encodings were checked for leakage: AUC on the training part ≈ AUC on the scored fold.
- **Price ratios, ZIP coordinates and bundle status are redundant** with existing columns and add a little noise.

## 2. Which household feature does the work?
Block B is the only block that helps, and by a wide margin (≈ +0.009 LightGBM, +0.007 CatBoost). Each of its four features is tested alone and left out in turn (LightGBM).

In [3]:
B = FE.BLOCKS['B_household']
rs = [base['lgb']] + [C.cv_score(f'fe_B_only_{f}_lgb', lab, KEEP + [f], notes=f'feature engineering: + {f} only') for f in B] \
     + [C.cv_score(f'fe_B_without_{f}_lgb', lab, KEEP + [x for x in B if x != f], notes=f'feature engineering: B without {f}') for f in B]
C.summary(rs, baseline=base['lgb']['name'])[['n_feats', 'auc_mean', 'Δ auc', 'se Δ', 'top_decile_prec']].style.format(fmt)

,n_feats,auc_mean,Δ auc,se Δ,top_decile_prec
name,,,,,
fe_base_lgb,37,0.6956,+0.0000,0.0000,27.8%
fe_B_only_DRV_AGE_35_50_lgb,38,0.6958,+0.0002,0.0004,27.9%
fe_B_only_DRV_AGE_SPREAD_lgb,38,0.6951,-0.0005,0.0003,27.8%
fe_B_only_N_YOUNG_DRIVERS_lgb,38,0.6950,-0.0006,0.0003,27.8%
fe_B_only_DRIVERS_PER_VEH_lgb,38,0.7042,+0.0086,0.0009,28.7%
fe_B_without_DRV_AGE_35_50_lgb,40,0.7045,+0.0088,0.0007,28.6%
fe_B_without_DRV_AGE_SPREAD_lgb,40,0.7051,+0.0095,0.0009,28.4%
fe_B_without_N_YOUNG_DRIVERS_lgb,40,0.7047,+0.0091,0.0008,28.2%
fe_B_without_DRIVERS_PER_VEH_lgb,40,0.6962,+0.0005,0.0005,27.8%


**`DRIVERS_PER_VEH` carries the whole gain.**
- On its own it adds +0.0086.
- Leaving it out of block B leaves +0.0005.
- The other three features (age band, age spread, number of young drivers) are neutral.

Below, a few alternative forms of the drivers-vs-vehicles relationship are compared, along with the conversion rate across the ratio.

In [4]:
lab['DRIVERS_PER_VEH_POL'] = lab.NUM_DRIVERS / lab.TOTAL_NUM_VEH  # same ratio with the policy-table vehicle count
lab['VEH_MINUS_DRIVERS'] = lab.VEH_N_ROWS - lab.NUM_DRIVERS
lab['MORE_DRIVERS_THAN_VEH'] = (lab.NUM_DRIVERS > lab.VEH_N_ROWS).astype('int8')
VARIANTS = {'ratio (vehicle table)': ['DRIVERS_PER_VEH'], 'ratio (policy table counts)': ['DRIVERS_PER_VEH_POL'],
            'vehicles − drivers': ['VEH_MINUS_DRIVERS'], 'more drivers than vehicles flag': ['MORE_DRIVERS_THAN_VEH'],
            'ratio + difference': ['DRIVERS_PER_VEH', 'VEH_MINUS_DRIVERS']}
rs = [base['lgb']] + [C.cv_score(f'fe_ratio_{n}_lgb', lab, KEEP + fs, notes=f'driver/vehicle variant: {n}') for n, fs in VARIANTS.items()]
display(C.summary(rs, baseline=base['lgb']['name'])[['auc_mean', 'Δ auc', 'se Δ', 'top_decile_prec']].style.format(fmt))
lab.groupby(pd.cut(lab.DRIVERS_PER_VEH, [0, .34, .5, .67, .99, 1.01, 1.5, 2.01, 7]), observed=True).CONVERT_IND.agg(
    conversion='mean', quotes='size').style.format({'conversion': '{:.1%}'})

,auc_mean,Δ auc,se Δ,top_decile_prec
name,,,,
fe_base_lgb,0.6956,+0.0000,0.0000,27.8%
fe_ratio_ratio (vehicle table)_lgb,0.7042,+0.0086,0.0009,28.7%
fe_ratio_ratio (policy table counts)_lgb,0.6948,-0.0008,0.0003,27.6%
fe_ratio_vehicles − drivers_lgb,0.7024,+0.0067,0.0010,29.1%
fe_ratio_more drivers than vehicles flag_lgb,0.7024,+0.0068,0.0009,28.8%
fe_ratio_ratio + difference_lgb,0.7036,+0.0079,0.0008,28.6%


,conversion,quotes
DRIVERS_PER_VEH,,
"(0.0, 0.34]",10.7%,9522
"(0.34, 0.5]",13.9%,5642
"(0.5, 0.67]",13.2%,3107
"(0.67, 0.99]",12.6%,1672
"(0.99, 1.01]",13.8%,4245
"(1.01, 1.5]",5.5%,2597
"(1.5, 2.01]",5.6%,1582
"(2.01, 7.0]",4.9%,1129


**Findings**
- **Households with more drivers than vehicles convert at ~5%**, vs 11–14% when there's at least one vehicle per driver. That's a 2.5× gap affecting ~18% of quotes.
- The trees struggled to build this from the two separate counts, so the explicit ratio unlocks it.
- **The vehicle-table ratio is the best form.** "More drivers than vehicles" and "vehicles − drivers" capture most of it (+0.0067).
- **The same ratio built from the policy table's vehicle count *hurts* (−0.0008).** That's more evidence that the policy-table vehicle counts are noise and the vehicle table is the real source (as in `02_feature_selection`).
- **For the presentation:** multi-driver households short on vehicles are a low-conversion segment. They likely face a higher price per car and need several people to agree; the call notes cite "other drivers need to agree".

## 3. Final configuration
**Selected 37 + the four household features (block B).**
- `DRIVERS_PER_VEH` carries nearly all of the gain.
- The other three features add a further +0.0014 for CatBoost (≈ 2.1 se, the keep threshold) and +0.0007 for LightGBM (≈ 1.2 se) on top of the ratio alone, so the whole block is kept.

None of these features uses the target, so the same feature list serves both models. The configuration is saved to `models/features_engineered.json` for tuning (`04_tuning`).

In [5]:
FINAL = KEEP + FE.BLOCKS['B_household']
ratio_only = {m: C.cv_score(f'fe_ratio_only_{m}', lab, KEEP + ['DRIVERS_PER_VEH'], model=m, notes='feature engineering: + DRIVERS_PER_VEH only') for m in base}
final = {m: C.cv_score(f'fe_final_{m}', lab, FINAL, model=m, notes='feature engineering: final (selected + household block)') for m in base}
display(pd.concat([C.summary([base[m], ratio_only[m], final[m]], baseline=base[m]['name']) for m in base]).style.format(fmt))
cfg = {m: {'features': FINAL, 'te_keys': [], 'make_model_te': False} for m in base}
json.dump(cfg | {'blocks_kept': ['B_household']}, open('features_engineered.json', 'w'), indent=1)
print(f'saved models/features_engineered.json ({len(FINAL)} features)')

,model,n_feats,rounds,auc_mean,Δ auc,se Δ,auc_se,top_decile_prec
name,,,,,,,,
fe_base_lgb,lgb,37,369,0.6956,+0.0000,0.0000,0.0033,27.8%
fe_ratio_only_lgb,lgb,38,359,0.7042,+0.0086,0.0009,0.0036,28.7%
fe_final_lgb,lgb,41,288,0.7049,+0.0092,0.0009,0.0036,28.7%
fe_base_cb,cb,37,424,0.7011,+0.0000,0.0000,0.0032,28.3%
fe_ratio_only_cb,cb,38,368,0.7064,+0.0053,0.0010,0.0033,28.8%
fe_final_cb,cb,41,323,0.7078,+0.0067,0.0011,0.0035,29.2%


saved models/features_engineered.json (41 features)


**Result**

| | Baseline (77 features) | Selected (37) | **Final (41)** |
|---|---|---|---|
| LightGBM CV AUC | 0.6940 | 0.6956 | **0.7049** |
| CatBoost CV AUC | 0.6986 | 0.7011 | **0.7078** |
| CatBoost top-decile precision | 28.0% | 28.3% | **29.2%** |

- Steps 1–2 together add **+0.011 AUC for LightGBM** and **+0.009 for CatBoost**, at the top of the +0.005 to +0.015 range the plan expected.
- **Next:** hyperparameter tuning (`04_tuning`) on this 41-feature set.